# Detected port infrastructure and regional operability risk

Exploratory overlay of Vili's 2024 infrastructure detections with stored coastal ERA5 operability results. This notebook **does not download ERA5 or alter the running retrieval process**.

Defaults: 2024 baseline; all candidate-site confidence levels on the map, including Likely natural; assessed High/Medium candidates for the comparisons; fully observed operability years; H3 resolution 4 (about 50-km-scale cells). The hexagons form a one-neighbour coastal band around frozen coastline anchors. They are **generalised regional zones, not additional observations or actual port downtime estimates**. A few H3 cells are pentagons.

Jetty size is the long axis of a detected oriented bounding box (OBB), **not surveyed jetty centreline length**. Inventory memberships determine unique structures. IDs were reindexed between supplied exports: raw OBBs are associated only after verifying class, centre distance <=1 m and confidence agreement. Ambiguous incompatible boxes raise an error.

Install the reproduction requirements in this notebook's Python environment if imports are missing.

In [16]:
%matplotlib inline
from pathlib import Path
from importlib import reload
import json
import h3
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from operability import analysis, infrastructure
reload(analysis)
reload(infrastructure)
from operability.workspace import Registry
from operability.products import atomic_csv, atomic_json
from port_operability import file_hash

WORKSPACE = Path('operability-workspace')
INFRASTRUCTURE = Path('literature/Vili/shapefiles/shapefile_port')
PERIOD = 'year_2024'
SCENARIO = 'baseline'
QUALITY = 'high_medium'  # Comparison selection; the map includes all confidence levels.
HEX_RESOLUTION = 4
MAX_DISTANCE_KM = 75.0
MIN_COVERAGE = 1.0  # Conservative: omit risk estimates with missing hours.
COUNTRIES = []  # Optional source-country filter, e.g. ['South Africa'].
WRITE_PRODUCTS = True

display(analysis.available_periods(WORKSPACE))
print('Period:', PERIOD, '| scenario:', SCENARIO, '| infrastructure quality:', QUALITY)

,period,method_signature,completed_points
2,year_2025,8891727a1fb184755152ec8faa825109a87236a257bdc5...,4002
1,year_2024,8891727a1fb184755152ec8faa825109a87236a257bdc5...,4002
0,year_2023,8891727a1fb184755152ec8faa825109a87236a257bdc5...,2432


Period: year_2024 | scenario: baseline | infrastructure quality: high_medium


## Load and verify inventory memberships and size proxies

A candidate site is a 1-km-scale inventory cluster, not necessarily a complete named port. Inventory counts include sites with zero detected jetties. Confidence refers to the candidate-site assessment, not independent validation of every detection. `QUALITY` selects candidates for comparisons and saved analysis tables, retaining the assessed High/Medium default. The map independently includes all 59,345 inventory candidates, including context-limited assessments and Likely natural sites (not confirmed ports), subject only to the optional country filter.

In [17]:
sites, jetties, inventory_qc = infrastructure.load_infrastructure(INFRASTRUCTURE, QUALITY)
display(pd.Series(inventory_qc, name='Inventory checks').to_frame())
display(sites.head())
display(jetties[['det_id', 'cluster_1km', 'year', 'obb_long_axis_m',
                 'obb_short_axis_m', 'source_match_distance_m']].head())

,Inventory checks
inventory_sites,59345
selected_sites,21686
selected_jetties,24820
jetties_excluded_by_site_selection,13725
box_source,raw jetty OBBs; inventory memberships prevent ...
box_matching,"same class, centre within 1 m, confidence with..."
maximum_box_match_distance_m,0.007722
length_interpretation,"OBB long-axis proxy, not surveyed centreline l..."


,cluster_id,n_members,n_basin,n_breakwater,n_container,n_jetty,n_liquid,centroid_lon,centroid_lat,assessment_status,port_confidence,jetty_count,jetty_obb_total_m,jetty_obb_median_m
0,c1km_0000000,3,1,0,0,2,0,63.477359,25.263305,assessed,Medium,2,332.799400,166.399700
1,c1km_0000001,1,0,1,0,0,0,62.462539,25.229214,assessed,Medium,0,0.000000,NaN
3,c1km_0000003,3,2,0,0,1,0,62.333541,25.114435,assessed,High,1,437.085794,437.085794
4,c1km_0000004,1,0,1,0,0,0,62.313658,25.126386,assessed,Medium,0,0.000000,NaN
5,c1km_0000005,1,0,1,0,0,0,62.331974,25.123934,assessed,Medium,0,0.000000,NaN


,det_id,cluster_1km,year,obb_long_axis_m,obb_short_axis_m,source_match_distance_m
0,4,c1km_0000000,2024,162.713317,65.604697,0.006135
1,5,c1km_0000000,2024,170.086084,104.633332,0.003314
2,19,c1km_0000006,2024,202.318887,138.306568,0.006061
3,20,c1km_0000003,2024,437.085794,104.782604,0.004252
4,29,c1km_0000007,2024,247.890392,92.099932,0.005637


## Associate regional risk without filling unsupported gaps

Every site is linked to the nearest **frozen ERA5 point**, including points without results. Missing, insufficient-coverage or more distant results remain unclassified; the algorithm never skips a missing nearest point to find a favourable available one.

Risk is computed at offshore locations. Nearest-neighbour association can cross straits or complex coastlines and does not model harbour sheltering. The distance and source point are retained for inspection. `COUNTRIES` filters by the **risk source's country**, not an independently established port-country identity.

In [18]:
with Registry(WORKSPACE, read_only=True) as registry:
    sources = registry.sites.copy()
summary = analysis.read_results(WORKSPACE, PERIOD)
if summary.empty:
    raise ValueError(f'No stored results for {PERIOD}; choose an available period above.')
summary = summary.loc[summary.scenario.eq(SCENARIO)].copy()
if summary.empty:
    raise ValueError(f'No stored results for scenario {SCENARIO}.')
linked_sites = infrastructure.link_risk(
    sites, sources, summary, 'centroid_lon', 'centroid_lat', MAX_DISTANCE_KM, MIN_COVERAGE)
if COUNTRIES:
    unknown = set(COUNTRIES) - set(sources.country)
    if unknown:
        raise ValueError(f'Unknown source countries: {sorted(unknown)}')
    linked_sites = linked_sites.loc[linked_sites.risk_country.isin(COUNTRIES)].copy()
selected_jetties = jetties.loc[jetties.cluster_1km.isin(linked_sites.cluster_id)].copy()
if linked_sites.empty:
    raise ValueError('No infrastructure sites for this selection.')
display(linked_sites.link_status.value_counts().rename('candidate sites').to_frame())
display(linked_sites[['cluster_id', 'risk', 'risk_site_id', 'risk_distance_km',
                      'coverage_fraction', 'link_status']].head())

,candidate sites
link_status,
matched,16970
beyond_distance_limit,4050
insufficient_coverage,508
no_result,158


,cluster_id,risk,risk_site_id,risk_distance_km,coverage_fraction,link_status
0,c1km_0000000,Low,7717,29.256796,1.0,matched
1,c1km_0000001,Low,7715,25.670644,1.0,matched
2,c1km_0000003,Low,7715,21.042935,1.0,matched
3,c1km_0000004,Low,7715,23.441577,1.0,matched
4,c1km_0000005,Low,7715,21.815226,1.0,matched


## Coastal hexagonal zones and infrastructure overlay

Colours use the nearest source to each hexagon **centre**; unsupported cells are grey. These are nearest-neighbour display zones, not smooth interpolation. Statistics use each candidate site's own coordinates, independently of hexagon resolution; near boundaries a site's popup may therefore differ from the underlying hexagon colour.

Translucent risk zones leave the coastline and background map visible. **Individual candidate sites** are split into High, Medium, Low and Likely natural confidence layers, all enabled by default. Open the foldable layers control at top right and click the confidence checkboxes to show/hide each group; colours and counts identify the groups. Click the **Regional operability risk** heading at bottom left to fold or unfold that legend. There is no port-aggregate layer. Confidence is the inventory assessment, not regional operability risk; context-limited status is retained in popups. Layer toggles affect only the map, not the comparison tables below. Use the same control to switch between **OpenStreetMap** and **Light-grey map (Esri)** or add optional Natural Earth outlines. Background tiles require an internet connection; zoom in for roads, labels and harbour context.

In [19]:
hexagons = infrastructure.coastal_hexagons(
    sources, summary, HEX_RESOLUTION, MAX_DISTANCE_KM, MIN_COVERAGE)
if COUNTRIES:
    hexagons = hexagons.loc[hexagons.risk_country.isin(COUNTRIES)].copy()
display(hexagons.link_status.value_counts().rename('hexagons').to_frame())
print('H3 cells:', len(hexagons), '| valid geometries:', hexagons.geometry.is_valid.all())
map_sites, _, _ = infrastructure.load_infrastructure(INFRASTRUCTURE, 'all')
map_sites = infrastructure.link_risk(
    map_sites, sources, summary, 'centroid_lon', 'centroid_lat', MAX_DISTANCE_KM, MIN_COVERAGE)
if COUNTRIES:
    map_sites = map_sites.loc[map_sites.risk_country.isin(COUNTRIES)].copy()
risk_infrastructure_map = infrastructure.infrastructure_map(
    hexagons, map_sites, analysis.load_land(WORKSPACE), HEX_RESOLUTION)
display(HTML(analysis.notebook_map_html(risk_infrastructure_map)))

,hexagons
link_status,
matched,12476
beyond_distance_limit,1691
insufficient_coverage,1001
no_result,50


H3 cells: 15218 | valid geometries: True


## Do higher-risk areas have more or larger detected jetties?

Compare **jetties per candidate site**, including zero-jetty sites, and the **distribution of individual OBB long-axis sizes**. Absolute counts alone are misleading when risk groups contain different numbers of ports. Overall cause availabilities overlap: they must not be added as exclusive downtime contributions.

These are descriptive associations, not causal findings. Candidate-site delineation, detection coverage/confidence, geography, cargo mix, port scale, sheltering and spatial clustering can confound comparisons. Use long-term periods once available and vary distance/coverage thresholds to check robustness.

In [ ]:
comparison = infrastructure.risk_comparison(linked_sites, selected_jetties)
if comparison.empty:
    raise ValueError('No supported infrastructure-risk associations for this selection.')
display(comparison.round(3))
supported = linked_sites.loc[linked_sites.link_status.eq('matched')]
jetty_risk = selected_jetties.merge(
    linked_sites[['cluster_id', 'risk', 'risk_site_id', 'risk_distance_km', 'link_status']],
    left_on='cluster_1km', right_on='cluster_id', how='left', validate='many_to_one')
figure, axes = plt.subplots(1, 2, figsize=(13, 4))
comparison.mean_jetties_per_site.plot.bar(ax=axes[0], color='#7842ad', rot=0)
axes[0].set(title='Detected jetties per selected candidate site',
            xlabel='Regional operability risk', ylabel='Mean count (zero-jetty sites included)')
length_groups = [risk for risk in infrastructure.RISK_ORDER
                 if jetty_risk.loc[jetty_risk.risk.eq(risk)].shape[0] > 0]
if length_groups:
    axes[1].boxplot([jetty_risk.loc[jetty_risk.risk.eq(risk), 'obb_long_axis_m']
                     for risk in length_groups], tick_labels=length_groups, showfliers=False)
    axes[1].set_yscale('log')
    axes[1].set(title='Detected jetty size proxy (not surveyed length)',
                xlabel='Regional operability risk', ylabel='OBB long axis (m, log scale)')
else:
    axes[1].text(.5, .5, 'No matched jetty boxes', ha='center', transform=axes[1].transAxes)
figure.tight_layout()
plt.show()

## Save clear, reusable analysis tables

CSV/gzip tables preserve associations, excluded-link statuses and source-match distances; provenance records selections and input hashes. They are regenerated for the current selection. No separate HTML map is saved here. Change `WRITE_PRODUCTS` to `False` for an entirely read-only run.

In [ ]:
if WRITE_PRODUCTS:
    output = WORKSPACE / 'infrastructure' / PERIOD / SCENARIO
    atomic_csv(output / 'site-risk.csv.gz', linked_sites)
    atomic_csv(output / 'jetty-risk.csv.gz', jetty_risk)
    atomic_csv(output / 'risk-comparison.csv', comparison.reset_index())
    input_files = [INFRASTRUCTURE / '1.raw/global_detections_raw.gpkg',
                   INFRASTRUCTURE / '3.evidence/members_enriched.csv',
                   INFRASTRUCTURE / '4.inventory/port_candidates_site_1km.csv']
    atomic_json(output / 'provenance.json', {
        'period': PERIOD, 'scenario': SCENARIO, 'quality': QUALITY,
        'countries': COUNTRIES, 'h3_resolution': HEX_RESOLUTION, 'h3_version': h3.__version__,
        'maximum_link_distance_km': MAX_DISTANCE_KM, 'minimum_coverage': MIN_COVERAGE,
        'inventory_checks': inventory_qc,
        'link_counts': {str(k): int(v) for k, v in linked_sites.link_status.value_counts().items()},
        'input_sha256': {str(path): file_hash(path) for path in input_files},
        'infrastructure_method_sha256': file_hash(Path(infrastructure.__file__)),
        'operability_method_signatures': sorted(summary.method_signature.unique().tolist()),
        'interpretation': 'Descriptive regional-exposure association; not protected-port downtime',
    })
    print('Saved analysis products:', output.resolve())
else:
    print('Read-only run: no analysis products written.')